In [40]:
import pandas as pd
import os
import numpy as np
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

EMBEDDING_PATH = os.path.join('mobilised_embeddings.npy')


In [29]:
df = pd.read_csv(os.path.join("..", "data", "msfc.csv"))

cols = ["Local Participant","VISITNUM", "Trial", "T25FW1L", "TFWMN1N"]

df = df[cols]

In [30]:
# Mapping
visit_map = {200: "t1", 210: "t2", 220: "t3", 230: "t4", 240: "t5"}
df["timestamp"] = df["VISITNUM"].map(visit_map)
df

,Local Participant,VISITNUM,Trial,T25FW1L,TFWMN1N,timestamp
0,25146,200,1,T,10.6,t1
1,24566,200,1,T,5.9,t1
2,25239,200,1,T,5.9,t1
3,25238,200,1,T,5.5,t1
4,25237,230,4,T,16.6,t4
...,...,...,...,...,...,...
2404,10376,240,5,T,11.6,t5
2405,10376,230,4,T,12.9,t4
2406,10376,220,3,T,11.4,t3
2407,10376,210,2,T,8.3,t2


In [31]:
# Finds rows where Local Participant and timestamp are repeated
duplicates = df[
    df.duplicated(subset=["Local Participant", "timestamp"], keep=False)
]

print(f"Total conflicting rows: {len(duplicates)}")
print(duplicates.sort_values(by=["Local Participant", "timestamp"]))

Total conflicting rows: 0
Empty DataFrame
Columns: [Local Participant, VISITNUM, Trial, T25FW1L, TFWMN1N, timestamp]
Index: []


In [34]:

df["timestamp"] = (df["timestamp"]).str.upper()

df["eid"] = (df["Local Participant"].astype(str) + "_" + df["timestamp"])

df["T25FW"] = df["TFWMN1N"]

tabular = df[["eid", "T25FW"]]

tabular

,eid,T25FW
0,25146_T1,10.6
1,24566_T1,5.9
2,25239_T1,5.9
3,25238_T1,5.5
4,25237_T4,16.6
...,...,...
2404,10376_T5,11.6
2405,10376_T4,12.9
2406,10376_T3,11.4
2407,10376_T2,8.3


In [39]:
def load_participant_embeddings(path):
    day_embeddings = np.load(path, allow_pickle=True).item()
    participants = {}
    for eid, days in day_embeddings.items():
        days = np.asarray(days, dtype=np.float32)
        norms = np.linalg.norm(days, axis=1)
        valid = np.isfinite(norms) & (norms > 0)  # a non-finite day gives a non-finite norm
        if valid.any():
            participants[str(eid)] = (days[valid] / norms[valid, None]).mean(axis=0)

    frame = pd.DataFrame.from_dict(participants, orient='index')
    frame.index.name = 'eid'
    frame.columns = [f'embedding_{column:03d}' for column in range(frame.shape[1])]
    return frame


embeddings = load_participant_embeddings(EMBEDDING_PATH)
embedding_columns = embeddings.columns.tolist()
tabular['eid'] = tabular['eid'].astype(str)
cohort = tabular.set_index('eid').join(embeddings, how='inner')
print(f'Participants with tabular data and embeddings: {len(cohort):,}')
cohort.reset_index()[['eid', 'T25FW']].head()

Participants with tabular data and embeddings: 994


/tmp/ipykernel_87960/3669687167.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tabular['eid'] = tabular['eid'].astype(str)


,eid,T25FW
0,24566_T1,5.9
1,25238_T1,5.5
2,25235_T4,7.5
3,25234_T3,30.1
4,25233_T1,7.5


In [41]:
# Define seed for reproducibility
SEED = 71

# 1. Identify embedding feature columns automatically
feature_cols = cohort.filter(regex=r"^embedding_").columns


# =========================================================================
# A. CONTINUOUS TARGETS: L2-Regularized Linear Regression (Ridge)
# =========================================================================
def evaluate_continuous_target(df, target_col, feature_cols, n_splits=5):
    """Fits L2-regularized Ridge regression using GroupKFold cross-validation."""
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]
    y = clean_df[target_col]
    groups = clean_df.index.astype(str).str.split("_").str[0]

    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(clean_df))

    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized linear regression
        model = Ridge(alpha=100.0, random_state=SEED)
        model.fit(X_train_scaled, y_train)

        oof_preds[test_idx] = model.predict(X_test_scaled)

    # Calculate overall dataset-level evaluation metrics
    r_stat, p_val = pearsonr(y, oof_preds)
    r2 = r2_score(y, oof_preds)
    mae = mean_absolute_error(y, oof_preds)
    rmse = root_mean_squared_error(y, oof_preds)

    print(f"=== Continuous Evaluation: {target_col} ===")
    print(f"Pearson's r : {r_stat:.3f} (p-value: {p_val:.2e})")
    print(f"R² Score    : {r2:.3f}")
    print(f"MAE         : {mae:.2f}")
    print(f"RMSE        : {rmse:.2f}\n")

    return oof_preds


# =========================================================================
# B. CATEGORICAL (BINARY & MULTI-CLASS) TARGETS: Class-Balanced L2 Logistic Regression
# =========================================================================
def evaluate_categorical_target(df, target_col, feature_cols, n_splits=5):
    """Fits class-balanced L2-regularized Logistic Regression (binary or multiclass)

    using StratifiedGroupKFold cross-validation.
    """
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]

    # Encode string categories into numeric labels
    le = LabelEncoder()
    y = le.fit_transform(clean_df[target_col].astype(str))
    classes = le.classes_
    num_classes = len(classes)

    # Extract base participant ID for group splitting
    groups = clean_df.index.astype(str).str.split("_").str[0]

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=SEED
    )
    oof_probs = np.zeros((len(clean_df), num_classes))

    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized Logistic Regression with balanced class weighting
        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=SEED,
        )
        clf.fit(X_train_scaled, y_train)

        oof_probs[test_idx] = clf.predict_proba(X_test_scaled)

    # Compute AUROC based on target cardinality
    if num_classes == 2:
        # Binary target (e.g. gender)
        overall_auroc = roc_auc_score(y, oof_probs[:, 1])
        print(
            f"=== Categorical Evaluation (Binary): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"AUROC : {overall_auroc:.4f}\n")
    else:
        # Multi-class target (e.g. healthstat)
        overall_auroc = roc_auc_score(
            y, oof_probs, multi_class="ovr", average="macro"
        )
        print(
            f"=== Categorical Evaluation (Multi-class): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"Macro OVR AUROC : {overall_auroc:.4f}")

        # Per-class AUROC breakdown
        for i, cls in enumerate(classes):
            cls_binary_y = (y == i).astype(int)
            cls_auroc = roc_auc_score(cls_binary_y, oof_probs[:, i])
            print(f"  └─ Class '{cls}' AUROC: {cls_auroc:.4f}")
        print()

    return oof_probs


# -------------------------------------------------------------------------
# Run linear probe evaluation pipeline on cohort
# -------------------------------------------------------------------------

oof_t25fw_preds = evaluate_continuous_target(cohort, "T25FW", feature_cols)

=== Continuous Evaluation: T25FW ===
Pearson's r : 0.594 (p-value: 7.79e-94)
R² Score    : 0.336
MAE         : 3.90
RMSE        : 6.28

